# Phase 1 EDA: Lahaina Climate-Risk

Exploratory data analysis of the parcel panel dataset.  
Run `make phase1` before executing this notebook.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

panel = pd.read_parquet('../data/final/panel.parquet')
print(f'Panel shape: {panel.shape}')
panel.head()

In [ ]:
# Summary statistics
summary_cols = [
    'log_price', 'sale_price', 'structure_sqft',
    'land_area_sqft', 'year_built', 'dist_to_fire_km'
]
available_cols = [c for c in summary_cols if c in panel.columns]
panel[available_cols].describe().round(3)

In [ ]:
# Histogram of log_price by treatment_band
fig, ax = plt.subplots(figsize=(9, 5))
if 'treatment_band' in panel.columns:
    for band in sorted(panel['treatment_band'].unique()):
        subset = panel[panel['treatment_band'] == band]['log_price'].dropna()
        ax.hist(subset, bins=40, alpha=0.5, label=band, density=True)
    ax.legend(title='Treatment Band')
else:
    panel['log_price'].hist(ax=ax, bins=40)
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel('log(Sale Price)')
ax.set_ylabel('Density')
ax.set_title('Distribution of log(Price) by Treatment Band')
plt.tight_layout()
plt.show()

In [ ]:
# Map of parcels colored by treatment_band
if 'lat' in panel.columns and 'lon' in panel.columns:
    unique_parcels = panel.drop_duplicates('parcel_id')
    gdf = gpd.GeoDataFrame(
        unique_parcels,
        geometry=gpd.points_from_xy(unique_parcels['lon'], unique_parcels['lat']),
        crs='EPSG:4326'
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    if 'treatment_band' in gdf.columns:
        gdf.plot(
            column='treatment_band', ax=ax, legend=True,
            legend_kwds={'loc': 'lower right', 'title': 'Treatment Band'},
            markersize=2, alpha=0.6, cmap='RdYlGn_r'
        )
    else:
        gdf.plot(ax=ax, markersize=2)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title('Maui Parcel Locations by Treatment Band')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()
else:
    print('lat/lon columns not available — skipping map cell')

In [ ]:
# Event-study figure (requires make phase1)
from pathlib import Path

es_path = Path('../results/event_study.csv')
if es_path.exists():
    es = pd.read_csv(es_path).sort_values('event_time')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.fill_between(es['event_time'], es['ci_lower'], es['ci_upper'],
                    alpha=0.2, color='#2166ac', label='95% CI')
    ax.plot(es['event_time'], es['att'], color='#2166ac',
            marker='o', markersize=3, linewidth=1.5, label='ATT')
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Event Time (months relative to fire)', fontsize=10)
    ax.set_ylabel('ATT (log price)', fontsize=10)
    ax.set_title('Event Study: Lahaina Fire Impact on Property Prices', fontsize=10)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('[Placeholder] Run `make phase1` to generate results/event_study.csv')